In [1]:
"""Module to use Real-ESRGAN model for super-resolution."""

from pathlib import Path
import numpy as np
import torch
from torch import nn as nn
from torch.nn import functional as F


class ResidualDenseBlock(nn.Module):
    """Residual Dense Block.

    Used in RRDB block in ESRGAN.

    Args:
        num_feat (int): Channel number of intermediate features.
        num_grow_ch (int): Channels for each growth.
    """

    def __init__(self, num_feat=64, num_grow_ch=32):
        super(ResidualDenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

        # initialization
        # default_init_weights([self.conv1, self.conv2, self.conv3, self.conv4, self.conv5], 0.1)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        # Empirically, we use 0.2 to scale the residual for better performance
        return x5 * 0.2 + x


class RRDB(nn.Module):
    """Residual in Residual Dense Block.

    Used in RRDB-Net in ESRGAN.

    Args:
        num_feat (int): Channel number of intermediate features.
        num_grow_ch (int): Channels for each growth.
    """

    def __init__(self, num_feat, num_grow_ch=32):
        super(RRDB, self).__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        # Empirically, we use 0.2 to scale the residual for better performance
        return out * 0.2 + x


class RRDBNet(nn.Module):
    """Networks consisting of Residual in Residual Dense Block, which is used
    in ESRGAN.

    ESRGAN: Enhanced Super-Resolution Generative Adversarial Networks.

    We extend ESRGAN for scale x2 and scale x1.
    Note: This is one option for scale 1, scale 2 in RRDBNet.
    We first employ the pixel-unshuffle (an inverse operation of pixelshuffle to reduce the spatial size
    and enlarge the channel size before feeding inputs into the main ESRGAN architecture.

    Args:
        num_in_ch (int): Channel number of inputs.
        num_out_ch (int): Channel number of outputs.
        num_feat (int): Channel number of intermediate features.
            Default: 64
        num_block (int): Block number in the trunk network. Defaults: 23
        num_grow_ch (int): Channels for each growth. Default: 32.
    """

    def __init__(self, num_in_ch, num_out_ch, scale=4, num_feat=64, num_block=23, num_grow_ch=32):
        super(RRDBNet, self).__init__()
        self.scale = scale
        if scale == 2:
            num_in_ch = num_in_ch * 4
        elif scale == 1:
            num_in_ch = num_in_ch * 16
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body = make_layer(RRDB, num_block, num_feat=num_feat, num_grow_ch=num_grow_ch)
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        # upsample
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        if self.scale == 2:
            feat = pixel_unshuffle(x, scale=2)
        elif self.scale == 1:
            feat = pixel_unshuffle(x, scale=4)
        else:
            feat = x
        feat = self.conv_first(feat)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat
        # upsample
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode="nearest")))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out


def make_layer(basic_block, num_basic_block, **kwarg):
    """Make layers by stacking the same blocks.

    Args:
        basic_block (nn.module): nn.module class for basic block.
        num_basic_block (int): number of blocks.

    Returns:
        nn.Sequential: Stacked blocks in nn.Sequential.
    """
    layers = []
    for _ in range(num_basic_block):
        layers.append(basic_block(**kwarg))
    return nn.Sequential(*layers)


def pixel_unshuffle(x, scale):
    """Pixel unshuffle.

    Args:
        x (Tensor): Input feature with shape (b, c, hh, hw).
        scale (int): Downsample ratio.

    Returns:
        Tensor: the pixel unshuffled feature.
    """
    b, c, hh, hw = x.size()
    out_channel = c * (scale**2)
    assert hh % scale == 0 and hw % scale == 0
    h = hh // scale
    w = hw // scale
    x_view = x.view(b, c, h, scale, w, scale)
    return x_view.permute(0, 1, 3, 5, 2, 4).reshape(b, out_channel, h, w)

class ESRGAN:

    def __init__(self, model_file: str=None, device: str = "cuda"):
        assert device in ["cuda", "cpu"], "device must be either 'cuda' or 'cpu'"
        self.model_path = Path(model_file).resolve() if model_file is not None else None
        self.device = torch.device(device)  # if you want to run on CPU, change 'cuda' -> cpu
        self.model = self._load_model()

    def _load_model(self):
        model = RRDBNet(
            num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4
        )
        
        if self.model_path is not None:

            # prefer to use params_ema
            loadnet = torch.load(self.model_path, map_location=torch.device("cpu"))

            if "params_ema" in loadnet:
                keyname = "params_ema"
            else:
                keyname = "params"
            model.load_state_dict(loadnet[keyname], strict=True)

        model.eval()
        model = model.to(self.device)
        return model

    def predict(self, img: np.ndarray) -> np.ndarray:
        """Generates a super-resolved image from a low-resolution image.

        Args:
            img (np.ndarray): image as 8-bit numpy array (RGB) (HxWx3)

        Returns:
            np.ndarray: super-resolved image as 8-bit numpy array (RGB) (HxWx3)
        """
        img = img * 1.0 / 255
        img = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()
        img_LR = img.unsqueeze(0)
        img_LR = img_LR.to(self.device)

        with torch.no_grad():
            output = self.model(img_LR).data.squeeze().float().cpu().clamp_(0, 1).numpy()
        # del img_LR
        output = np.transpose(output, (1, 2, 0))
        output = (output * 255.0).round()
        output = output.astype(np.uint8)
        return output

In [2]:
model = ESRGAN()
total_params = sum(p.numel() for p in model.model.parameters())
print(total_params)

16697987


In [3]:
from thop import profile

# shape must match the model forward signature (batch included)
input = torch.randn(1, 3, 64, 64).to(next(model.model.parameters()).device)

macs, params = profile(model.model, inputs=(input,), verbose=False)
flops = 2 * macs   # convert MACs -> FLOPs (if you want multiply+add=2 FLOPs)
print(f"MACs: {macs:,}, Params: {params:,}, FLOPs: {flops:,}")

MACs: 73,428,369,408.0, Params: 16,697,987.0, FLOPs: 146,856,738,816.0


In [4]:
import torch, time

def benchmark(model, inp, n_warmup=10, n_iter=50):
    model = model.eval().cuda()
    inp = inp.cuda()
    # warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(inp)
    # measure
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(n_iter):
            _ = model(inp)
    torch.cuda.synchronize()
    end = time.time()
    return (end - start) / n_iter

# example
device = torch.device('cuda')
model.model.to(device)

inp = torch.randn(16, 3, 256, 256, device=device)
t = benchmark(model.model, inp)
print(f"Average latency per forward: {t*1000:.2f} ms")
print(t)


Average latency per forward: 1347.68 ms
1.3476848649978637


In [5]:
t

1.3476848649978637